In [ ]:
import pandas as pd
import numpy as np

import os, sys, gc, time, warnings, pickle, psutil, random

from tqdm import tqdm
from math import ceil
from sklearn.preprocessing import LabelEncoder


In [ ]:
grid_df = pd.read_parquet('train_val.parquet')
grid_df = grid_df.dropna(subset=['demand'])

In [ ]:
grid_df

In [ ]:
print(grid_df.columns)

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Khởi tạo cột out_of_stock với giá trị mặc định là 0
grid_df['out_of_stock'] = 0

# Lấy danh sách các sản phẩm duy nhất
prods = list(grid_df.id.unique())

# Ngưỡng gap: 90 ngày
GAP_THRESHOLD = 90

# Duyệt qua từng sản phẩm
for prod_id in tqdm(prods, desc="Processing products"):
    # Lọc dữ liệu của sản phẩm hiện tại
    m = grid_df.id == prod_id
    idx = grid_df.loc[m, 'demand'].index
    sales = grid_df.loc[m, 'demand'].values  # Chuỗi doanh số

    # Tạo mảng nhị phân: 1 nếu doanh số = 0, 0 nếu doanh số > 0
    no_sales = (sales == 0).astype(int)  # Ví dụ: [0, 2, 0, 0, 1] → [1, 0, 1, 1, 0]

    # Tìm các gap (chuỗi ngày không bán liên tiếp)
    gap_lengths = []
    current_gap = 0
    out_of_stock = np.zeros(len(sales), dtype=int)  # Mảng kết quả

    for i in range(len(no_sales)):
        if no_sales[i] == 1:  # Ngày không bán
            current_gap += 1
            if i == len(no_sales) - 1 and current_gap >= GAP_THRESHOLD:
                # Nếu gap kéo dài đến ngày cuối và >= 90 ngày
                out_of_stock[i - current_gap + 1:i + 1] = 1
        else:  # Ngày có bán
            if current_gap >= GAP_THRESHOLD:
                # Nếu gap vừa kết thúc và dài >= 90 ngày, đánh dấu tất cả ngày trong gap là out_of_stock
                out_of_stock[i - current_gap:i] = 1
            current_gap = 0  # Reset gap khi gặp ngày có bán

    # Gán kết quả vào cột out_of_stock trong grid_df
    grid_df.loc[m, 'out_of_stock'] = out_of_stock

# Hiển thị một phần kết quả
print(grid_df[['id', 'demand', 'out_of_stock']].head())

In [ ]:
grid_df.to_parquet('grid_df_out_of_stock.parquet', index=False, engine='pyarrow')